Za projekt iz prompt inžinjeringa sam koristio promptove za oblikovanje recepta da izvodi određene zadatke. Koristio sam  Google Generative AI koji nudi besplatni tier s do 1 milijun tokena mjesečno.

In [ ]:
import google.generativeai as genai


API_KEY = "TVOJ_API_KLJUC"


try:
    genai.configure(api_key=API_KEY)
    print("API ključ uspješno postavljen!")
except Exception as e:
    print(f"Greška pri postavljanju API ključa: {e}")

API ključ uspješno postavljen!


S import dodajemo gemini ai pod aliasom genai. Definiramo varijablu API_KEY za spremanje gemini api ključa. S blokom try pokušajem konfigurirati gemini model s ključem ako ne uspije except ispisuje grešku.

In [ ]:
try:
    model = genai.GenerativeModel('gemini-1.5-pro-latest')
    print("Gemini model uspješno inicijaliziran!")
except Exception as e:
    print(f"Greška pri inicijalizaciji Gemini modela: {e}")

Gemini model uspješno inicijaliziran!


Sa try pokušajemo pozvati instancu gemini modela i spremiti u varijablu model. Ako ne uspije s blokom except će se print dio ispisati i e se koristi kako bi ispisali grešku.

In [ ]:
def get_completion(prompt):
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Greška pri generiranju odgovora: {e}")
        return None



get_completion funkcija pokušaje generirati odgovor na dani prompt, ako ne uspije ispisuje grešku.

In [ ]:
import requests

def dohvat_recepata():
    url = "http://pzi202024.studenti.sum.ba/backend/api/dohvatipredjelo"
    response = requests.get(url)
    if response.status_code == 200:
        print("Recepti uspješno dohvaćeni!")
        return response.json()
    else:
        print("Greška pri dohvatu recepata.")
        return None


recepti = dohvat_recepata()



primjer_recepta = recepti['recepti']['data'][1]
print("Primjer jednog recepta:",primjer_recepta)


Recepti uspješno dohvaćeni!
Primjer jednog recepta: {'recept_id': 7, 'naziv_recepta': 'Otvoreni sendvič s pečenom bundevom', 'slika': 'storage/slike/9q97UpzboSwX66eVXavKnHE1PwQ2gODFXUptdcfg.jpg', 'vrstajela': 'Predjelo', 'created_at': '2024-06-25T15:54:55.000000Z', 'korisnicko_ime': 'Mario676'}


S import request uvozimo biblioteku za rad sa zahtjevima poput get i post. Funkcija dohvat_recepata dohvaća podatke s get api zahtijeva i vraća ih u json formatu. Tu funkciju spremamo kasnije u varijablu recepti i ispisujemo prvi recept kojeg sam iskoristio kao primjer.

In [ ]:
link_slike = 'http://pzi202024.studenti.sum.ba/backend/' + primjer_recepta['slika']
print(link_slike)


http://pzi202024.studenti.sum.ba/backend/storage/slike/9q97UpzboSwX66eVXavKnHE1PwQ2gODFXUptdcfg.jpg


In [ ]:
import base64

def get_image_data(image_url):
   #Funkcija za preuzimanje slike s navedenoga linka
    response = requests.get(image_url)
    return {
        "mime_type": "image/jpeg",
        "data": base64.b64encode(response.content).decode("utf-8")
    }

U Google ai dokumentaciji je navedeno kako genai radi samo sa slikama u formatu pillow i base64 ne radi s linkovima slika pa sam morao preuzeti sliku i formatirati ju,  te ju vratiti u oblik koji genai prepoznaje za rad sa slikama. Kod return je naveden oblik rječnika koji ima kljuc mime_type: u kojemu je definiran format slike u ovome slučaju radi se o jpeg formatu slike. Kod data je uloga utf8 da vrši enkodiranje tj pretvara iz binarnoga oblika u tekstualni.

In [ ]:
slika = get_image_data(link_slike)

In [ ]:
print(slika)

{'mime_type': 'image/jpeg', 'data': '...'}


In [ ]:
prompt = [
    "Opiši što vidiš na slici.",
    slika
]



response = get_completion(prompt)

print(response)


Slika prikazuje dvije bruskete s namazom od sira, kriškama pečenog povrća (vjerojatno tikve) i peršinom. Bruskete su postavljene na komad papira za pečenje, a ispod se nazire drvena daska. U pozadini je zamućena slika više komada povrća, vjerojatno onog korištenog za bruskete, te noža i nečega što nalikuje na krem sir. Pozadina je tirkizne boje.  Slika je snimljena izbliza te djeluje vrlo privlačno i ukusno.



Ovdje sam testirao da li radi opisivanje slike.

In [ ]:
print(primjer_recepta)

{'recept_id': 7, 'naziv_recepta': 'Otvoreni sendvič s pečenom bundevom', 'slika': 'storage/slike/9q97UpzboSwX66eVXavKnHE1PwQ2gODFXUptdcfg.jpg', 'vrstajela': 'Predjelo', 'created_at': '2024-06-25T15:54:55.000000Z', 'korisnicko_ime': 'Mario676'}


Recept ima svoje atribute pomoću kojih će genai generirati tekst kao kratak opis recepta.


In [ ]:
def opis_recepta(recipe):
    prompt =  [ f"""
-Analiziraj sljedeći recept i napiši jednu rečenicu koja ga najbolje opisuje.
-Rečenica treba biti kreativna i ograniči ju na minimalno 8 a maksimalno 14 riječi.
-Kod kreiranja opisa recepta prije toga analiziraj sliku tako da izdvojiš je li prikladna te generiraj opis na osnovu slike  i taj opis slike uzmi u obzir za sto bolji opis recepta.
    - Daj odgovor u formatu json:
    -Slika:
    -Opis slike:
    - Opis:
-Kod kreiranja kreativnoga opisa uzmi u obzir
    - naziv recepta: {recipe['naziv_recepta']}
    - vrsta jela: {recipe['vrstajela']}


Neka ti temperatura odgovora bude 0.7 to jest 70 posto mogućnosti da ti odogovor bude različit.


    """ , slika ]
    return get_completion(prompt)



prompt = opis_recepta(primjer_recepta)

print(prompt)


```json
{
  "Slika": true,
  "Opis slike": "Na slici su dva otvorena sendviča s pečenom bundevom, kremastim sirom i peršinom na drvenoj dasci.",
  "Opis": "Sočni zalogaji jeseni: hrskavi kruh s pečenom bundevom i kremastim sirom."
}
```


Ovdje sam naveo genai da mi opise sliku na osnovu prethodno dohvaćeni naziva i vrste jela te da uz to uzme i u obzir opis slike. Kako genai nema kao open ai parametar Temperature naveo sam mu unutar prompta čisto da simulira odnosno pokusa dati drugačije odgovore.

In [ ]:
def fetch_recipes():
    url = "http://pzi202024.studenti.sum.ba/backend/api/recept/4"
    response = requests.get(url)
    if response.status_code == 200:
        print("Komentari uspješno dohvaćeni!")
        return response.json()
    else:
        print("Greška pri dohvatu podataka o receptu.")
        return None

# Dohvati recept
recipes = fetch_recipes()

# Provjeri jeli recept sadrži podatke
if recipes:
  komentari_recepta = recipes['komentari']
  print(komentari_recepta)


Komentari uspješno dohvaćeni!
[{'sadržaj': 'Ovaj recept za čokoladni tortu izgleda jednostavno, ali ukusno! Sviđa mi se što koristi tri biskvita, što će sigurno dati dobru teksturu. Čokoladni fil zvuči predivno, posebno s kombinacijom bijele i crne čokolade. Jedino što me malo brine je vrijeme pečenja – 20-25 minuta po biskvitu znači da će priprema trajati malo duže, ali vjerujem da će rezultat biti vrijedan truda. Definitivno ću probati ovaj recept!', 'datum_Komentara': '2025-03-11 13:00:16', 'idrecepta': 4, 'created_at': '2025-03-11T12:00:16.000000Z', 'updated_at': '2025-03-11T12:00:16.000000Z', 'korisnicko_ime': 'superadmin1'}, {'sadržaj': 'Novi komentar!😊', 'datum_Komentara': '2024-06-29 11:43:38', 'idrecepta': 4, 'created_at': '2024-06-29T09:43:38.000000Z', 'updated_at': '2024-06-29T09:43:38.000000Z', 'korisnicko_ime': 'Mario676'}]


Ovdje sam definirao funkciju za dohvat recepta s api poziva gdje je cilj bio dohvatiti postojeće komentare recepta za dalji rad sa prompt.

In [ ]:
for komentar in  komentari_recepta:
  print(komentar['sadržaj'])

Ovaj recept za čokoladni tortu izgleda jednostavno, ali ukusno! Sviđa mi se što koristi tri biskvita, što će sigurno dati dobru teksturu. Čokoladni fil zvuči predivno, posebno s kombinacijom bijele i crne čokolade. Jedino što me malo brine je vrijeme pečenja – 20-25 minuta po biskvitu znači da će priprema trajati malo duže, ali vjerujem da će rezultat biti vrijedan truda. Definitivno ću probati ovaj recept!
Novi komentar!😊


In [ ]:
print(len(komentari_recepta))

2


In [ ]:
for komentar in komentari_recepta:
  prompt = f"""
 Tvoj zadatak je da sljedeće komentare
  - 1.Ispravi pismene greške
  -2. Odredi jeli komentar pozitivan ili negativan
  - 3. Klasificiraj u obliku jeli prikladan za recept odnosno provjeri je li recept ima neprimjerene riječi i je li cijeli komentar vezan za recept
  - Prikladan :
  - Ocjena komentara:
  - 4. Ako komentar je duži od 15 riječi skrati komentar ali da bude što sličniji originalu
  - 5. ako je komentar kraći od 15 riječi da ga proširiš ali opet neka je što sličniji komentaru.


  -6. Vrati odgovor u obliku json u formatu
  <Originalni komentar: >
  <Ispravljeni komentar:>
  <Prikladan: >
  <Ocjena: >


  Komentar: {komentar['sadržaj']}

  """
  response = get_completion(prompt)
  print(response)


```json
{
  "Originalni komentar": "Ovaj recept za čokoladni tortu izgleda jednostavno, ali ukusno! Sviđa mi se što koristi tri biskvita, što će sigurno dati dobru teksturu. Čokoladni fil zvuči predivno, posebno s kombinacijom bijele i crne čokolade. Jedino što me malo brine je vrijeme pečenja – 20-25 minuta po biskvitu znači da će priprema trajati malo duže, ali vjerujem da će rezultat biti vrijedan truda. Definitivno ću probati ovaj recept!",
  "Ispravljeni komentar": "Recept za čokoladnu tortu djeluje jednostavno, ali ukusno! Sviđa mi se kombinacija tri biskvita i čokoladnog fila s bijelom i crnom čokoladom. Vrijeme pečenja je malo duže, ali vjerujem da će se isplatiti. Isprobat ću!",
  "Prikladan": true,
  "Ocjena": "pozitivan"
}
```

```json
{
  "Originalni komentar": "Novi komentar!😊",
  "Ispravljeni komentar": "Ovo je novi komentar! 😊 Sjajno je isprobati nove recepte.",
  "Prikladan": true,
  "Ocjena": "pozitivan"
}
```



U ovome kodu sa za svaki komentar odradio prompt kojemu je cilj da pravilno oblikuje komentar i odredi određena stanja je li primjeren i je li pozitivan.

In [ ]:
# 1. Dobro napisan recept
recept1 = {
    "naziv_recepta": "Torta od višanja",
    "opis_recepta": "Ukusna i sočna torta s višnjama, idealna za sve ljubitelje voćnih slastica. Kombinacija mekanog biskvita, osvježavajućih višanja i lagane kreme čini ovu tortu savršenim desertom.",
    "sastojci_recepta": [
        "4 jaja",
        "150 g šećera",
        "100 g brašna",
        "50 g kakaa",
        "1 prašak za pecivo",
        "500 ml slatkog vrhnja",
        "200 g mascarpone sira",
        "300 g višanja",
        "2 žlice šećera u prahu"
    ],
    "primjena_recepta": [
        "Umutiti jaja sa šećerom dok smjesa ne postane pjenasta.",
        "Dodati brašno, kakao i prašak za pecivo te lagano promiješati.",
        "Pecite u kalupu na 180°C oko 25 minuta.",
        "Ohlađeni biskvit prerežite na pola.",
        "Umutiti slatko vrhnje sa šećerom u prahu, dodati mascarpone i promiješati.",
        "Na prvu polovicu biskvita staviti kremu i višnje, prekriti drugom polovicom i premazati ostatkom kreme.",
        "Ohladiti prije posluživanja."
    ]
}

# 2. Recept s neprimjerenim riječima i vulgarnostima
recept2 = {
    "naziv_recepta": "Torta od višanja",
    "opis_recepta": "Ova torta je jeb**no najbolja stvar koju ćete ikada napraviti! Ako ne znate kako da napravite tortu, evo vam prilike da se ne osramotite pred gostima. Garantiram da će vam svi lizati tanjure.",
    "sastojci_recepta": [
        "3 glupa jajeta",
        "200 g šećera (ili koliko vam se *ebe)",
        "100 g brašna, valjda",
        "500 ml nekog vrhnja, šta ja znam kojeg",
        "200 g višanja, ne pitajte me dal' svježih ili smrznutih"
    ],
    "primjena_recepta": [
        "Ma samo sve strpajte u neku zdjelu i miksajte dok ne postane fino.",
        "Ispecite to na nekoj temperaturi, recimo 180°C, ili dok ne vidite da nije sirovo.",
        "Kad je gotovo, ubacite višnje i neki krem, ako vam se da.",
        "Ako ste uspjeli, svaka čast. Ako niste, hebiga, probajte opet!"
    ]
}

# 3. Nedovljno dobro opisan recept
recept3 = {
    "naziv_recepta": "Torta od višanja",
    "opis_recepta": "Ukusna voćna torta.",
    "sastojci_recepta": [
        "4 jaja",
        "100 g brašna",
        "višnje"
    ],
    "primjena_recepta": [
        "Umiješati sastojke.",
        "Ispeći u pećnici.",
        "Dodati višnje."
    ]
}
#Primjer ne dovoljno dobro napisanoga recepta
recept4 = {
    "naziv_recepta": "Nešto fino",
    "opis_recepta": "Super je.",
    "sastojci_recepta": [],
    "primjena_recepta": ["Napraviti."]
}



In [ ]:
def analiziraj_recept(recept):
    prompt = f"""
    Ti si AI asistent kojem je zadatak da odradi sljedeće korake kod analize recepta koje ću ti navesti:

    -1. Oblikuj i sortiraj cijeli recept tako da dani tekst formatiraš u oblik koji sam ti ispod naveo također uzmi u obzir da recept može imati korake navedene u krivim dijelovima pa pazi i na te detalje:
    - Naziv recepta: <kratak naziv recepta>
    - Opis: <u 3 do 5 rečenica da je opisan recept>
    - Sastojci: <navedeni sastojci recepta poželjno je da su navedene i količine>
    - Priprema: <navedeni koraci kako se pravi>

    -2. Provjeri je li recept prikladan:
    - ako recept ima nedozvoljene riječi (vulgarnosti, rečenice koje nemaju veze s receptom)
    - ako recept ima dosta nedostataka (ako veći dio recepta nedostaje i nije dobro napisan što ćeš vidjeti iz 'Ispis nedostatka recepta')
    - vrati Prikladan: false
    - u suprotnome ispiši:
    - Prikladan: true

    -3. Ispravi gramatičke greške i ako nedostaju detaljniji opisi, koraci ili sastojci, daj komentar:
    - Ispis nedostatka recepta: <ovdje navedi što nije dobro napisano o receptu odnosno što nedostaje>
    - ako je recept dobro napisan onda ovako odgovori:
      - Ispis nedostatka recepta:<'Recept je dobro napisan.'>

    -4. Ako recept nije dobro napisan, ako je moguće, pokušaj popraviti recept na osnovu cijelog teksta koji si proveo kroz prethodno navedene korake.
    - Nastoji kod poboljšanja recepta da bude što sličniji originalnome, odnosno nemoj previše mijenjati, samo ispravi nedostatke.

    -5. Uvijek vrati odgovor u listi:
      [
        {{
          "prijedlozi": [<lista prijedloga>],
          "nedostaci": [<lista problema>],
          "prikladan": boolean  <true ako je recept prikladan>
        }},
        "html"  <Poboljšani HTML recept (SAMO ako je prikladan)>
      ]

    Primjer odgovora za prihvatljiv recept:

    [
      {{
        "prijedlozi": ["Dodajte vreme pečenja za kore"],
        "nedostaci": ["Nedostaju sastojci za glazuru"],
        "prikladan": true
      }},
      "<h1>Torta...</h1><!-- Nedostatak: Nedostaje glazura -->"
    ]

    Primjer odgovora za neprikladan recept:
    [
      {{
        "prijedlozi": ["Počnite ispočetka", "Dodajte sve sastojke"],
        "nedostaci": ["Sadrži vulgarnosti", "Nedostaje 95% sadržaja"],
        "prikladan": false
      }},
      ""
    ]

    Recept: {recept}
    """

    response = get_completion(prompt)
    return response



In [ ]:
recept1_response = analiziraj_recept(recept1)
print(recept1_response)

```json
[
  {
    "prijedlozi": [],
    "nedostaci": ["Recept je dobro napisan."],
    "prikladan": true
  },
  "<h1>Torta od višanja</h1>\n<p>Ukusna i sočna torta s višnjama, idealna za sve ljubitelje voćnih slastica. Kombinacija mekanog biskvita, osvježavajućih višanja i lagane kreme čini ovu tortu savršenim desertom.</p>\n<h2>Sastojci:</h2>\n<ul>\n<li>4 jaja</li>\n<li>150 g šećera</li>\n<li>100 g brašna</li>\n<li>50 g kakaa</li>\n<li>1 prašak za pecivo</li>\n<li>500 ml slatkog vrhnja</li>\n<li>200 g mascarpone sira</li>\n<li>300 g višanja</li>\n<li>2 žlice šećera u prahu</li>\n</ul>\n<h2>Priprema:</h2>\n<ol>\n<li>Umutiti jaja sa šećerom dok smjesa ne postane pjenasta.</li>\n<li>Dodati brašno, kakao i prašak za pecivo te lagano promiješati.</li>\n<li>Pecite u kalupu na 180°C oko 25 minuta.</li>\n<li>Ohlađeni biskvit prerežite na pola.</li>\n<li>Umutiti slatko vrhnje sa šećerom u prahu, dodati mascarpone i promiješati.</li>\n<li>Na prvu polovicu biskvita staviti kremu i višnje, prekri

In [ ]:
recept2_response = analiziraj_recept(recept2)
print(recept2_response)

```json
[
  {
    "prijedlozi": [
      "Uklonite vulgarne izraze.",
      "Dodajte precizne količine i vrste sastojaka.",
      "Napišite detaljne korake pripreme."
    ],
    "nedostaci": [
      "Recept sadrži vulgarne izraze.",
      "Nedostaju precizne količine i vrste sastojaka.",
      "Koraci pripreme su nejasni i nepotpuni."
    ],
    "prikladan": false
  },
  ""
]
```



In [ ]:
recept3_response = analiziraj_recept(recept3)
print(recept3_response)

```json
[
  {
    "prijedlozi": [
      "Dodajte količinu višanja",
      "Dodajte količinu šećera",
      "Dodajte temperaturu i vrijeme pečenja",
      "Detaljnije opišite korake pripreme"
    ],
    "nedostaci": [
      "Nedostaju detalji o količini nekih sastojaka (npr. višnje, šećer).",
      "Nedostaju detalji o temperaturi i vremenu pečenja.",
      "Koraci pripreme su previše općeniti."
    ],
    "prikladan": true
  },
  "<h1>Torta od višanja</h1>\n<p>Ukusna voćna torta.</p>\n<h2>Sastojci:</h2>\n<ul>\n<li>4 jaja</li>\n<li>100 g brašna</li>\n<li>višnje (količina?)</li>\n<li>šećer (količina?)</li>\n</ul>\n<h2>Priprema:</h2>\n<ol>\n<li>Umiješati sastojke. (Koje sastojke? Kako ih umiješati? Treba li mikser, pjenjača...?)</li>\n<li>Ispeći u pećnici. (Na kojoj temperaturi? Koliko dugo?)</li>\n<li>Dodati višnje. (Svježe ili iz kompota? Prije ili poslije pečenja?)</li>\n</ol><!-- Nedostaci: Detaljnije opisati korake, količinu sastojaka i temperaturu/vrijeme pečenja -->"
]
```



In [ ]:
recept4_response = analiziraj_recept(recept4)
print(recept4_response)

```json
[
  {
    "prijedlozi": [
      "Dodajte detaljniji opis recepta (3-5 rečenica).",
      "Dodajte sastojke.",
      "Proširite upute za pripremu."
    ],
    "nedostaci": [
      "Opis recepta je prekratak.",
      "Nedostaju sastojci.",
      "Priprema recepta je nepotpuna."
    ],
    "prikladan": false
  },
  ""
]
```



Ovdje sam testirao prompt za analizu recepta dao sam 4 primjera gdje prvi recept je dobro napisan, drugi sadrži ne primjerene riječi i treći nedostaju detaljniji opisi recepta i četvrti je nedovoljno dobro napisan ima puno nedostataka. U sva 4 slučaja sam dobio željene odgovore od prompta kod kojeg je bilo cilj prikazati kako će recept s ne prikladnim riječima postaviti prikladan na false a ostala 2 primjera će staviti da su prikladni kod 4. primjera je cilj bio da prepozna kada recept nije dovoljno dobro napisan i postavio je prikladan na false.

In [ ]:
import json

def oblikuj_format(odgovor: str):

    try:

        clean_str = odgovor.strip().replace('```json', '').replace('```', '')
        parsed_data = json.loads(clean_str)


        if isinstance(parsed_data, list) and len(parsed_data) == 2:
            return {"analiza": parsed_data[0], "html": parsed_data[1]}
        else:
            return {"error": "Neispravan format odgovora"}

    except json.JSONDecodeError as e:
        return {"error": f"Greška pri parsiranju JSON-a: {e}"}
    except Exception as e:
        return {"error": f"Neočekivana greška: {e}"}

Želio sam sa display(HTML()) prikazati dobijeni html dio pa iz razloga što genai uvijek vraća u tekstualnome obliku morao sam izlaz iz stringa pretvoriti u listu te sam iz liste uzeo podijelu gdje je prvi dio analiza a drugi dio html kod.

In [ ]:

rezultat = oblikuj_format(recept1_response)

if "error" not in rezultat:
    print("Analiza:", rezultat["analiza"])
    print("HTML:", rezultat["html"])
else:
    print("Greška:", rezultat["error"])

Analiza: {'prijedlozi': [], 'nedostaci': ['Recept je dobro napisan.'], 'prikladan': True}
HTML: <h1>Torta od višanja</h1>
<p>Ukusna i sočna torta s višnjama, idealna za sve ljubitelje voćnih slastica. Kombinacija mekanog biskvita, osvježavajućih višanja i lagane kreme čini ovu tortu savršenim desertom.</p>
<h2>Sastojci:</h2>
<ul>
<li>4 jaja</li>
<li>150 g šećera</li>
<li>100 g brašna</li>
<li>50 g kakaa</li>
<li>1 prašak za pecivo</li>
<li>500 ml slatkog vrhnja</li>
<li>200 g mascarpone sira</li>
<li>300 g višanja</li>
<li>2 žlice šećera u prahu</li>
</ul>
<h2>Priprema:</h2>
<ol>
<li>Umutiti jaja sa šećerom dok smjesa ne postane pjenasta.</li>
<li>Dodati brašno, kakao i prašak za pecivo te lagano promiješati.</li>
<li>Pecite u kalupu na 180°C oko 25 minuta.</li>
<li>Ohlađeni biskvit prerežite na pola.</li>
<li>Umutiti slatko vrhnje sa šećerom u prahu, dodati mascarpone i promiješati.</li>
<li>Na prvu polovicu biskvita staviti kremu i višnje, prekriti drugom polovicom i premazati ostatk

In [ ]:

from IPython.display import display, HTML


print(display(HTML(rezultat["html"])))


None


In [ ]:
prompt = f"""
Tvoj zadatak je da prevedeš recept sa hrvatskoga jezika na engleski.
želim samo da mi html dio vratiš ne trebam nikakva objašnjenja.


  Recept: {rezultat["html"]}

"""
response = get_completion(prompt)

print(response)


```html
<h1>Cherry Cake</h1>
<p>A delicious and moist cake with cherries, ideal for all fruit dessert lovers. The combination of soft sponge cake, refreshing cherries and light cream makes this cake a perfect dessert.</p>
<h2>Ingredients:</h2>
<ul>
<li>4 eggs</li>
<li>150 g sugar</li>
<li>100 g flour</li>
<li>50 g cocoa powder</li>
<li>1 baking powder</li>
<li>500 ml heavy cream</li>
<li>200 g mascarpone cheese</li>
<li>300 g cherries</li>
<li>2 tablespoons powdered sugar</li>
</ul>
<h2>Preparation:</h2>
<ol>
<li>Whisk eggs with sugar until the mixture becomes frothy.</li>
<li>Add flour, cocoa powder and baking powder and mix gently.</li>
<li>Bake in a mold at 180°C for about 25 minutes.</li>
<li>Cut the cooled sponge cake in half.</li>
<li>Whip the heavy cream with powdered sugar, add mascarpone and mix.</li>
<li>Place the cream and cherries on the first half of the sponge cake, cover with the second half and spread with the remaining cream.</li>
<li>Chill before serving.</li>
</ol>
`

U ovome dijelu sam koristio prevođenje gdje sam tekst recepta preveo sa hrvatskoga na engleski koristeći genai prompt.

In [ ]:
print(display(HTML(response)))

None


Cilj projekta je bio prikazati kako radi promt inženjering. Koristio sam api pozive od jedne aplikacije domaći recepti čisto kako bi prikazao kako funkcionira prompt koristeći api kao input. Koristio sam elementa prompta od sažimanja zaključivanja prevođenja korake prompta sam navodio rednim brojevima kao koraci te sam <> koristio kod opisa koraka i {} sam koristio za input varijable u promptu.